# Mapping the Potential Destructive Power of Wildfires Using Machine Learning
---
## Module 2: *Data Merging*
##### Version Number: 3.0
---
### Contents  
> 1. *Merge sample points with weather data*
> 2. *Spatial Join of Nearest Sampling Locations with Fire Damage Datasets*
> 3. *Export File*
---
### Notes
- Integrate wildfire impact data with daily weather data from extracted from gridMET.
### Inputs
- Daily Weather Readings - `gridmet_final.csv` 
- Wildfire Damage Data - `clean_fires.csv` (cleaned in module 1)
- California Mesh Sampling Grid - `sampling_points.csv` (built in appendix A) 
---
### Outputs  
- `model_fire_pop_income.csv` Cleaned weather dataset merged with fire damage severity, population and mean income data
---
### User Created Dependencies  

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

# Show basic facts about a dataframe
from src.data_utils import basic_explore

# basic health checks after a merge
from src.data_utils import post_merge_check

---
### Third Party Dependencies

In [2]:
# Data handling
import pandas as pd
import numpy as np
import geopandas as gpd
import datetime as dt

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
from shapely.geometry import Point

---

### Data Loading and Exploration

In [3]:
weather = pd.read_csv('../data/raw/gridmet_final.csv')
fire_data = pd.read_csv("../data/processed/fire_data.csv")
samples = pd.read_csv("../data/raw/sampling_points.csv")

## 1. Merge sample points with weather data

In [4]:
# Load geometry
samples['geometry'] = [Point(xy) for xy in zip(samples['Sample_Longitude'], samples['Sample_Latitude'])]
samples_gdf = gpd.GeoDataFrame(samples, geometry='geometry', crs="EPSG:4326")

## generate unique identifier
samples_gdf['Sample_ID'] = range(1, len(samples_gdf) + 1)

## fix error from ArcGIS
samples_gdf['County'] = samples_gdf['County'].replace('South Coast Valleys', 'San Diego')

In [5]:
# Merge on BOTH station and date
samples_daily = weather.merge(
    samples_gdf,
    on=['Sample_ID'],
    how='left'
)

### Reorganize so Sample ID and Date are first two columns
#samples_daily.insert(0, 'Date', samples_daily.pop('Date'))
#samples_daily.insert(0, 'Sample_ID', samples_daily.pop('Sample_ID'))

In [6]:
post_merge_check (samples_daily, weather)

Premerged shape:  (446340, 22)
Merged shape:  (446340, 39)
Duplicates before merge:  0
Duplicates after merge:  0
NA values before merge:  0
NA values after merge:  0


## 2. Spatial Join of Nearest Sampling Locations with Fire Damage Datasets

To prepare for feature engineering, we spatially join each fire location to its nearest sampling point. This enables us to associate daily environmental conditions with each fire based on geographic proximity, rather than relying solely on county or administrative boundaries.

In [7]:
fire_data['geometry'] = [Point(xy) for xy in zip(fire_data['Fire_Longitude'], fire_data['Fire_Latitude'])]
fire_data_gdf = gpd.GeoDataFrame(fire_data, geometry='geometry', crs="EPSG:4326")

samples_daily_gdf = gpd.GeoDataFrame(samples_daily, geometry='geometry', crs="EPSG:4326")

## Project to Equal area projection for more accuracy
samples_projected = samples_daily_gdf.to_crs(3310)
fire_data_projected = fire_data_gdf.to_crs(3310)

In [8]:
## Buffer distance in meters, Sampling points are located approximately 46000 meters apart
buffer_dist = 36000

In [9]:
## Initialize variables to track total damage and number of fires associated with each point
samples_projected['fire_count'] = 0
fire_data_projected['total_fire_damage'] = 0.0

#### Main loop of Spatial Join algorithm
- Loop through all dates in fire damage dataset
- Save all fires associated with the current date
    - If no fires, move to next date
- Load sampling points associated with current date
- Create a buffer, sized as defined earlier, around each fire for current date
- Intersection spatial join of sampling points and buffered fires
- Aggregate in case of multiple fires in a buffered range
    - Total estimated cost is summed for all fires in range
- Assign samples to dataframe

In [10]:
for dt in fire_data_projected['Date'].unique():
    
    # Fires on this date
    fires_today = fire_data_projected[fire_data_projected['Date'] == dt]
    if fires_today.empty:
        continue

    # Samples on this date
    samples_today = samples_projected[samples_projected['Date'] == dt]
    if samples_today.empty:
        continue

    # Create buffers around each fire
    fires_buffered = fires_today.copy()
    fires_buffered['geometry'] = fires_buffered.buffer(buffer_dist)

    # Spatial join: find samples within fire buffers
    joined = gpd.sjoin(samples_today, fires_buffered, how='left', predicate='intersects')

    # Aggregate counts and total damage per sample
    grouped = joined.groupby('Sample_ID').agg(
        fire_count=('Estimated Damage', 'count'),
        total_fire_damage=('Estimated Damage', 'sum'),
        acres = ('FinalAcres', 'sum')
    ).fillna(0)

    # Assign values back to main dataframe
    samples_projected.loc[samples_projected['Date'] == dt, 'fire_count'] = \
        samples_projected.loc[samples_projected['Date'] == dt, 'Sample_ID'].map(grouped['fire_count']).fillna(0)
    samples_projected.loc[samples_projected['Date'] == dt, 'total_fire_damage'] = \
        samples_projected.loc[samples_projected['Date'] == dt, 'Sample_ID'].map(grouped['total_fire_damage']).fillna(0)
    samples_projected.loc[samples_projected['Date'] == dt, 'acres'] = \
        samples_projected.loc[samples_projected['Date'] == dt, 'Sample_ID'].map(grouped['acres']).fillna(0)

In [11]:
post_merge_check(samples_projected, samples_daily)

Premerged shape:  (446340, 39)
Merged shape:  (446340, 42)
Duplicates before merge:  0
Duplicates after merge:  0
NA values before merge:  0
NA values after merge:  8996


In [12]:
samples_projected.isna().sum()

Sample_ID                                                         0
Date                                                              0
Burning Index                                                     0
Energy Release Component                                          0
Actual Evapotranspiration                                         0
100-hour Dead Fuel Moisture                                       0
1000-hour Dead Fuel Moisture                                      0
Precipitation                                                     0
Maximum Relative Humidity                                         0
Minimum Relative Humidity                                         0
Specific Humidity                                                 0
Solar Radiation                                                   0
Daily Minimum Air Temperature                                     0
Daily Maximum Air Temperature                                     0
Vapor Pressure Deficit                          

NA values are for buffered areas without any fires in range within the date range. Fill these values with 0. 

In [13]:
samples_projected['total_fire_damage'] = samples_projected['total_fire_damage'].fillna(0)
samples_projected['acres'] = samples_projected['acres'].fillna(0)

#### Clean Dataframe

In [14]:
## sort dataframe by date
samples_projected = samples_projected.sort_values(by='Date')

## reset index
samples_projected = samples_projected.reset_index(drop=True)

---

## 3. Export File

In [16]:
samples_projected.to_csv('../data/processed/samples_projected.csv',index=False)
print("All datasets saved successfully to ../data/processed/")

All datasets saved successfully to ../data/processed/
